# Principal components and factor analysis: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(8)

### 1. Six indicators driven by two latent factors (loadings 0.8, rotated by 35 degrees)

In [ ]:
# 1. Six indicators driven by two latent factors (loadings 0.8, rotated by 35 degrees)
n, p, k = 400, 6, 2
L = np.zeros((p, k))
for j in range(p):
    L[j, j % k] = 0.8
c, s = np.cos(0.6), np.sin(0.6)
L = L @ np.array([[c, s], [-s, c]])
F = rng.standard_normal((n, k))
X = F @ L.T + np.sqrt(1 - (L ** 2).sum(1)) * rng.standard_normal((n, p))
X = pd.DataFrame(X, columns=[f"X{j + 1}" for j in range(p)])
Z = StandardScaler().fit_transform(X)
R = np.corrcoef(Z.T)

### 2. Eigenvalues and the scree

In [ ]:
# 2. Eigenvalues and the scree
w, V = np.linalg.eigh(R)
w, V = w[::-1], V[:, ::-1]
print("eigenvalues:", np.round(w, 2), "| Kaiser (>1):", int((w > 1).sum()), "| cumulative share:", np.round(np.cumsum(w) / p, 2))

### 3. PCA loadings = eigenvector * sqrt(eigenvalue); check against sklearn

In [ ]:
# 3. PCA loadings = eigenvector * sqrt(eigenvalue); check against sklearn
pca = PCA(n_components=2).fit(Z)
load_pca = V[:, :2] * np.sqrt(w[:2])
print("\nsklearn explained variance ratio:", np.round(pca.explained_variance_ratio_, 3), "| from eigenvalues:", np.round(w[:2] / p, 3))
print("PCA loadings (unrotated):\n", np.round(load_pca, 2))
print("communalities (PCA, 2 components):", np.round((load_pca ** 2).sum(1), 2))

### 4. Two-variable case used in the numerical problem: eigenvalues are 1 +- r

In [ ]:
# 4. Two-variable case used in the numerical problem: eigenvalues are 1 +- r
r2 = 0.6
print("\n2x2 correlation r = 0.6: eigenvalues", np.linalg.eigvalsh(np.array([[1, r2], [r2, 1]]))[::-1], "| PC1 loading sqrt((1+r)/2) =", round(np.sqrt((1 + r2) / 2), 4))

### 5. Varimax rotation (Kaiser normalisation), written out

In [ ]:
# 5. Varimax rotation (Kaiser normalisation), written out
def varimax(A, iters=100, tol=1e-8):
    A = A.copy(); h = np.sqrt((A ** 2).sum(1))[:, None]; A = A / h
    p_, k_ = A.shape; Rm = np.eye(k_); d = 0
    for _ in range(iters):
        B = A @ Rm
        u, sv, vt = np.linalg.svd(A.T @ (B ** 3 - B @ np.diag((B ** 2).sum(0)) / p_))
        Rm = u @ vt; dn = sv.sum()
        if dn < d * (1 + tol): break
        d = dn
    return (A @ Rm) * h
rot = varimax(load_pca)
print("\nvarimax loadings:\n", np.round(rot, 2))
print("communalities unchanged by rotation:", np.round((rot ** 2).sum(1), 2))

### 6. Factor analysis (maximum likelihood) with sklearn

In [ ]:
# 6. Factor analysis (maximum likelihood) with sklearn
fa = FactorAnalysis(n_components=2, rotation="varimax", random_state=0).fit(Z)
print("\nFA loadings (varimax):\n", np.round(fa.components_.T, 2))
print("FA uniquenesses:", np.round(fa.noise_variance_, 2), "| true uniquenesses:", np.round(1 - (L ** 2).sum(1), 2))

## Expected output

Running all cells above should print:

```
eigenvalues: [2.49 2.16 0.39 0.38 0.32 0.26] | Kaiser (>1): 2 | cumulative share: [0.41 0.78 0.84 0.9  0.96 1.  ]

sklearn explained variance ratio: [0.415 0.36 ] | from eigenvalues: [0.415 0.36 ]
PCA loadings (unrotated):
 [[-0.68 -0.55]
 [ 0.63 -0.63]
 [-0.66 -0.6 ]
 [ 0.65 -0.59]
 [-0.69 -0.54]
 [ 0.54 -0.67]]
communalities (PCA, 2 components): [0.77 0.8  0.79 0.77 0.77 0.75]

2x2 correlation r = 0.6: eigenvalues [1.6 0.4] | PC1 loading sqrt((1+r)/2) = 0.8944

varimax loadings:
 [[-0.88 -0.02]
 [ 0.11 -0.89]
 [-0.89 -0.07]
 [ 0.15 -0.86]
 [-0.88 -0.01]
 [ 0.02 -0.87]]
communalities unchanged by rotation: [0.77 0.8  0.79 0.77 0.77 0.75]

FA loadings (varimax):
 [[ 0.81 -0.02]
 [-0.08  0.84]
 [ 0.84  0.03]
 [-0.12  0.8 ]
 [ 0.81 -0.02]
 [ 0.    0.77]]
FA uniquenesses: [0.35 0.28 0.3  0.35 0.35 0.4 ] | true uniquenesses: [0.36 0.36 0.36 0.36 0.36 0.36]
```

## Practice exercises

Try these in the cells below.

1. Generate data with 2 true factors. How many eigenvalues exceed 1? Does the scree plot show the elbow?
2. Increase the unique noise. How do the communalities and the first eigenvalue change?
3. Tick "Rotate (varimax)" and compare the loading pattern before and after.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
